In [ ]:
from simulator import Simulator
from simulator.config import Color
from simulator.helpers import clean
from simulator.helpers.coordinates import ENUPose, GRAPose
from simulator.planner import AutoPlan, GuidedPlan, Plan
from simulator.visualizer import (
    QGC,
    Gazebo,
    GazMarker,
    NoVisualizer,
    QGCMarker,
    SimVehicle,
)

clean()

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241,alt=0,heading=0) 
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading) 

base_homes= ENUPose.list([  # east, north, up, heading
    (5, 5, 0, 0),
    (10, 0, 0, 0),
    (-5, -10, 0, 0),
    (-15, 0, 0, 0),
    (0, -20, 0, 0),

])
side_lens = (5, 7, 4, 3, 2)

In [ ]:
sysids = [1,2,3,4,5]
colors = 2*[Color.BLUE]+3*[Color.GREEN]
alt = 5

vehs:list[SimVehicle] = []
for sysid, base_home, color, slen in zip(sysids, base_homes, colors, side_lens):
    auto_plan = AutoPlan.square_traj(
        side_len=slen,
        alt=alt,
        name="simple_auto_plan",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=base_home,
    )
    
    guided_plan = GuidedPlan.square_traj(
                                    side_len=slen, 
                                    alt=alt,
                                    enu_origin=enu_origin,
                                    relative_home=base_home)
    
    veh = SimVehicle.from_relative(
        sysid=sysid,
        gcs_name=f'{color.name}_{color.emoji}',
        plan=guided_plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=Plan.create_square_path(slen),
        model="iris",
    )
    vehs.append(veh)
    

In [ ]:
gaz= Gazebo(gra_origin,world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_gaz = GazMarker(name="origin",
                    group="origin",
                    pos=enu_origin.unpose(),
                    color=Color.WHITE)
gaz.markers.append(origin_gaz)

In [ ]:
qgc= QGC(gra_origin)
origin_qgc = QGCMarker(name="origin",
                pos=gra_origin.unpose(),
                color=Color.WHITE)
qgc.markers.append(origin_qgc)

In [ ]:
novis = NoVisualizer(gra_origin)

In [ ]:
simulator = Simulator(
	visualizer=gaz,
	terminals=['gcs'],
	verbose=1,
)
for veh in vehs:
    simulator.add_vehicle(veh)

simulator.show()

In [ ]:
orac = simulator.launch()
orac.run()